Testing out pynaple and dandi env on this dataset: https://dandiarchive.org/dandiset/001695/0.260319.2023

In [1]:
from dandi.dandiapi import DandiAPIClient

DANDISET_ID = "001695"
VERSION_ID = "0.260319.2023"

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, VERSION_ID)
    # dandiset is a RemoteDandiset pinned to your published version

A newer version (0.79.0) of dandi/dandi-cli is available. You are using 0.78.0


In [2]:
# All assets, sorted by path
for asset in dandiset.get_assets(order="path"):
    print(asset.path, asset.identifier, asset.size)

# Just one subject's sessions — this is your "iterate over subfiles" loop
for asset in dandiset.get_assets_with_path_prefix("sub-M01/", order="path"):
    print(asset.path, asset.size)

# Or glob, e.g. only behavior+ecephys sessions across all subjects
for asset in dandiset.get_assets_by_glob("**/*behavior+ecephys.nwb"):
    print(asset.path)

sub-M01/sub-M01_ses-20240308T100000_ecephys.nwb 48507af3-9fb8-4ec2-ba63-3e19b2703352 70720864
sub-M01/sub-M01_ses-20240312T100000_behavior+ecephys.nwb 52b551c4-2a2f-4a87-82f5-3d804b7cb82a 187350474
sub-M01/sub-M01_ses-20240313T100000_behavior+ecephys.nwb 6d733831-afbf-44c2-8c46-7b3550f5e672 125772852
sub-M01/sub-M01_ses-20240314T100000_behavior+ecephys.nwb 80edb3f8-ff28-4694-9b55-7ef98b4d43dc 134620736
sub-M01/sub-M01_ses-20240318T100000_behavior+ecephys.nwb b597de93-315f-477c-ac3d-1b87f0f3bbe9 190033984
sub-M02/sub-M02_ses-20240226T100000_ecephys.nwb a0c10954-8d63-4114-b91f-e5b0560ee2bc 79311908
sub-M02/sub-M02_ses-20240307T100000_ecephys.nwb 9a3df357-88e8-48f8-b52c-4abc28e7da03 114018584
sub-M02/sub-M02_ses-20240312T100000_behavior+ecephys.nwb 1e4d5403-a8cc-4814-a904-7aff57f8cc4d 61347328
sub-M02/sub-M02_ses-20240313T100000_behavior+ecephys.nwb b9876bef-9508-4329-b5ed-8d9da2a3e201 98347742
sub-M02/sub-M02_ses-20240314T100000_behavior+ecephys.nwb 0a41ffc1-8216-4663-8842-14e4222742ad 1

In [6]:
from pathlib import Path

# notebook is in /notebooks/CumulantDynamicsGuiltByAssociation
# storage container is two levels up, then into storage/
DOWNLOAD_DIR = Path("../../storage/dandi_downloads").resolve()
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)  # create if missing
print("Downloads go to:", DOWNLOAD_DIR)

Downloads go to: /storage/dandi_downloads


In [7]:
from dandi.dandiapi import DandiAPIClient

DANDISET_ID = "001695"
VERSION_ID = "0.260319.2023"

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, VERSION_ID)
    # list sub-M01 sessions with sizes (size is free from the listing, no download)
    sessions = [(a.path, a.identifier, a.size)
                for a in dandiset.get_assets_with_path_prefix("sub-M01/", order="path")]

for path, _id, size in sessions:
    print(f"{size/1e6:7.1f} MB  {path}")

# pick the smallest one so the download/analysis is quick
target_path = min(sessions, key=lambda t: t[2])[0]
print("\nPicked:", target_path)


   70.7 MB  sub-M01/sub-M01_ses-20240308T100000_ecephys.nwb
  187.4 MB  sub-M01/sub-M01_ses-20240312T100000_behavior+ecephys.nwb
  125.8 MB  sub-M01/sub-M01_ses-20240313T100000_behavior+ecephys.nwb
  134.6 MB  sub-M01/sub-M01_ses-20240314T100000_behavior+ecephys.nwb
  190.0 MB  sub-M01/sub-M01_ses-20240318T100000_behavior+ecephys.nwb

Picked: sub-M01/sub-M01_ses-20240308T100000_ecephys.nwb


In [8]:
# stream just enough to see what's inside — no full download
# (as_readable is blob-only; these .nwb files are blobs)
import h5py, pynwb

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, VERSION_ID)
    asset = dandiset.get_asset_by_path(target_path)
    with asset.as_readable().open() as f:
        with h5py.File(f) as h5:
            with pynwb.NWBHDF5IO(file=h5, mode="r") as io:
                nwb = io.read()
                print(nwb)  # prints the NWB structure (acquisition, units, intervals, etc.)

/storage/conda_envs/pynapple_dandi_env/lib/python3.11/site-packages/numcodecs/__init__.py:106: DeprecationWarning: crc32c usage is deprecated since numcodecs v0.16.4. It is recommended to install google_crc32c instead.
  from numcodecs.checksum32 import CRC32, Adler32, JenkinsLookup3


root pynwb.file.NWBFile at 0x140436586918928
Fields:
  devices: {
    Neuropixels 2.0 <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    ElectrodeGroup1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  experimenter: ['Mihály Vöröslakos']
  file_create_date: [datetime.datetime(2026, 1, 13, 21, 59, 2, 489026, tzinfo=tzoffset(None, -18000))]
  identifier: name
  institution: NYU
  intervals: {
    Odor Stimulus <class 'pynwb.epoch.TimeIntervals'>
  }
  lab: Buzsáki Lab
  session_description: Head Fix Session with Odor Stimulation
  session_start_time: 2024-03-08 10:00:00+00:00
  subject: subject pynwb.file.Subject at 0x140436585123984
Fields:
  age: P6W/P15W
  age__reference: birth
  description: Wild-type mouse
  sex: M
  species: Mus musculus
  strain: C57BL/6J
  subject_id: M01

  timestamps_reference_time: 2024-03-08 10:00:00+00:00
  units: units <class 'pynwb.misc.Units'>



In [10]:
asset_out = DOWNLOAD_DIR / target_path  # preserves sub-M01/... subpath

# create the sub-M01/ (and any nested) parent dirs — download() won't do this itself
asset_out.parent.mkdir(parents=True, exist_ok=True)

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, VERSION_ID)
    asset = dandiset.get_asset_by_path(target_path)
    asset.download(asset_out)  # blocks until complete

print("Downloaded to:", asset_out)

Downloaded to: /storage/dandi_downloads/sub-M01/sub-M01_ses-20240308T100000_ecephys.nwb


In [11]:
import pynapple as nap

# pynapple has a native NWB loader; it exposes units, epochs, tsd, etc.
data = nap.load_file(str(asset_out))
print(data)  # shows the available objects in this NWB file

# grab spike times if this session has sorted units
units = data["units"]           # TsGroup of spike trains
print(f"\n{len(units)} units")
print(units)

# simplest useful analysis: firing rate per unit over the whole recording
rates = units.rate             # spikes / sec per unit
print("\nMean firing rates (Hz):")
print(rates)

sub-M01_ses-20240308T100000_ecephys
┍━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━┑
│ Keys          │ Type        │
┝━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━┥
│ units         │ TsGroup     │
│ Odor Stimulus │ IntervalSet │
┕━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━┙

541 units
  Index      rate  cell_type           cell_area      firing_rate    ab_ratio    burstIndex_Mizuseki2012    cv2    maxWaveformCh    troughToPeak    acg_tau_decay    acg_tau_rise    thetaModulationIndex    x_position_probe    y_position_probe
-------  --------  ------------------  -----------  -------------  ----------  -------------------------  -----  ---------------  --------------  ---------------  --------------  ----------------------  ------------------  ------------------
      1   9.57618  Wide Interneuron    RSC                   9.58        0.21                       0      0.64              145            0.46            16.11           25.83                   -0.07                 309                3240
      2   6.06982  Wide Interneuron